In [ ]:
from dolfin import *
import numpy as np
import matplotlib.pyplot as plt
from fenics import project
import torch
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
import os
import argparse

def extract_edges(mesh):
    # Extract the element-to-node connectivity from the mesh
    mesh_connectivity = mesh.cells()  # Get element-to-node mapping
    edges = set()
    for element in mesh_connectivity:
        for i in range(len(element)):
            for j in range(i + 1, len(element)):
                edges.add(tuple(sorted((element[i], element[j]))))
    edge_index = list(edges)
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    bidir_edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
    return bidir_edge_index, mesh_connectivity

def extract_node_masses(u_, du, rho):
    # Extract node wise lumped masses
    m_mass = rho * inner(u_, du) * dx
    M = assemble(m_mass)
    diag = Vector()
    M.init_vector(diag, 0)  
    M.get_diagonal(diag)
    lumped_masses_per_dof = diag.get_local()
    lumped_masses_nodes = lumped_masses_per_dof.reshape(-1,3) 
    return lumped_masses_nodes   

def extract_mesh_bc(bc, V):
    # Get constrained DOF indices
    bc_dofs_dict = bc.get_boundary_values()
    bc_dof_indices = np.array(list(bc_dofs_dict.keys()), dtype=int)

    # Number of components
    num_components = V.num_sub_spaces() or V.ufl_element().value_size()
    num_dofs = V.dim()
    num_nodes = num_dofs // num_components

    # Map DOF indices to node indices
    node_indices = bc_dof_indices // num_components

    is_fixed = np.zeros(num_nodes, dtype=int)
    is_fixed[node_indices] = 1
    node_scalar = torch.tensor(is_fixed, dtype=torch.float).reshape(-1,1)    
    return node_scalar

def plot_check(time, u_tip, energies, save_path):
    if MPI.comm_world.rank == 0:
        fig, ax = plt.subplots(2, 1, figsize=(8, 6))

        # Tip displacement
        ax[0].plot(time, u_tip)
        ax[0].set_xlabel("Time")
        ax[0].set_ylabel("Tip displacement")
        ax[0].set_title("Tip Displacement Evolution")

        # Energies
        ax[1].plot(time, energies)
        ax[1].legend(("Elastic", "Kinetic", "Damping", "Total"))
        ax[1].set_xlabel("Time")
        ax[1].set_ylabel("Energies")
        ax[1].set_title("Energies Evolution")

        plt.tight_layout()
        plt.savefig(save_path)
        print('plot saved to :', save_path)
        plt.show()

def save_graphs(lst_graph_tstep, save_path):
    gph_dataloader = DataLoader(lst_graph_tstep, batch_size=1, shuffle=False)
    torch.save(gph_dataloader, save_path)
    print(f'saved graphs(dataloader) to {save_path}')


# ---------------- NEW HELPER: evaluate_u_at_vertices ----------------
def evaluate_u_at_vertices(u, mesh):
    """
    Evaluate the solution 'u' (VectorFunctionSpace) at each vertex
    in mesh.coordinates().

    Returns a NumPy array (num_vertices, 3) with the displacement
    for each vertex.  (In 3D.)
    """
    coords = mesh.coordinates()
    nverts = coords.shape[0]
    out = np.zeros((nverts, 3), dtype=float)

    for i in range(nverts):
        out[i] = u(*coords[i])  # Evaluate the solution at vertex i

    return out


def fea_simulation(L, W, D, NL, NW, ND, elastic_params, density, damping_params, newmark_params,
                   initial_force=1.0, cutoff_time_factor=1/5, total_time=4.0, num_steps=50, mode='train'):

    parameters["form_compiler"]["cpp_optimize"] = True
    parameters["form_compiler"]["optimize"] = True

    # Elastic parameters
    E, nu = elastic_params
    mu = Constant(E / (2.0 * (1.0 + nu)))
    lmbda = Constant(E * nu / ((1.0 + nu) * (1.0 - 2.0 * nu)))

    # Mass density
    rho = Constant(density)

    # Damping
    eta_m_val, eta_k_val = damping_params
    eta_m = Constant(eta_m_val)
    eta_k = Constant(eta_k_val)

    # Generalized-alpha
    alpha_m_val, alpha_f_val = newmark_params
    alpha_m = Constant(alpha_m_val)
    alpha_f = Constant(alpha_f_val)
    gamma = Constant(0.5 + alpha_f - alpha_m)
    beta = Constant((gamma + 0.5)**2 / 4.)

    # Mesh
    mesh = BoxMesh(Point(0., 0., 0.), Point(L, W, D), NL, NW, ND)

    def left(x, on_boundary):
        return near(x[0], 0.) and on_boundary

    def right(x, on_boundary):
        return near(x[0], L) and on_boundary

    V = VectorFunctionSpace(mesh, "CG", 1)
    Vsig = TensorFunctionSpace(mesh, "DG", 0)

    bidir_edge_index, mesh_connectivity = extract_edges(mesh)

    du = TrialFunction(V)
    u_ = TestFunction(V)

    u = Function(V, name="Displacement")
    u_old = Function(V)
    v_old = Function(V)
    a_old = Function(V)

    boundary_subdomains = MeshFunction("size_t", mesh, mesh.topology().dim() - 1)
    boundary_subdomains.set_all(0)
    force_boundary = AutoSubDomain(right)
    force_boundary.mark(boundary_subdomains, 3)
    dss = ds(subdomain_data=boundary_subdomains)

    # Clamped BC
    zero = Constant((0.0, 0.0, 0.0))
    bc = DirichletBC(V, zero, left)
    node_scalar = extract_mesh_bc(bc, V)

    T = total_time
    Nsteps = num_steps
    dt = Constant(T / Nsteps)

    # Load expression
    p0 = initial_force
    cutoff_Tc = T * cutoff_time_factor
    p = Expression(("0", "t <= tc ? p0 * t / tc : 0", "0"), t=0, tc=cutoff_Tc, p0=p0, degree=0)

    # Stress
    def sigma(r):
        return 2.0 * mu * sym(grad(r)) + lmbda * tr(sym(grad(r))) * Identity(len(r))

    def m(uA, uB):
        return rho * inner(uA, uB) * dx

    def k(uA, uB):
        return inner(sigma(uA), sym(grad(uB))) * dx

    def c(uA, uB):
        return eta_m*m(uA,uB) + eta_k*k(uA,uB)

    def Wext(uA):
        return dot(uA, p)*dss(3)

    def update_a(uNEW, uOLD, vOLD, aOLD, ufl=True):
        if ufl:
            dt_ = dt
            beta_ = beta
        else:
            dt_ = float(dt)
            beta_ = float(beta)
        return (uNEW - uOLD - dt_*vOLD)/(beta_*dt_**2) - (1-2*beta_)/(2*beta_)*aOLD

    def update_v(aNEW, uOLD, vOLD, aOLD, ufl=True):
        if ufl:
            dt_ = dt
            gamma_ = gamma
        else:
            dt_ = float(dt)
            gamma_ = float(gamma)
        return vOLD + dt_*((1-gamma_)*aOLD + gamma_*aNEW)

    def update_fields(uNEW, uOLD, vOLD, aOLD):
        u_vec, u0_vec = uNEW.vector(), uOLD.vector()
        v0_vec, a0_vec = vOLD.vector(), aOLD.vector()
        a_vec = update_a(u_vec, u0_vec, v0_vec, a0_vec, ufl=False)
        v_vec = update_v(a_vec, u0_vec, v0_vec, a0_vec, ufl=False)
        vOLD.vector()[:] = v_vec
        aOLD.vector()[:] = a_vec
        uOLD.vector()[:] = uNEW.vector()

    def avg(xOLD, xNEW, alpha):
        return alpha*xOLD + (1-alpha)*xNEW

    a_new = update_a(du, u_old, v_old, a_old, ufl=True)
    v_new = update_v(a_new, u_old, v_old, a_old, ufl=True)
    res = m(avg(a_old,a_new,alpha_m), u_) + c(avg(v_old,v_new,alpha_f), u_) \
          + k(avg(u_old,du,alpha_f), u_) - Wext(u_)

    from ufl import lhs, rhs
    a_form = lhs(res)
    L_form = rhs(res)

    K, res_assembled = assemble_system(a_form, L_form, bc)
    solver = LUSolver(K, "mumps")
    solver.parameters["symmetric"] = True

    time = np.linspace(0, T, Nsteps + 1)
    u_tip = np.zeros((Nsteps + 1,))
    energies = np.zeros((Nsteps + 1, 4))
    E_damp = 0
    sig = Function(Vsig, name="sigma")

    lumped_masses_nodes = extract_node_masses(u_, du, rho)

    def local_project(v, V_, uOut=None):
        dv = TrialFunction(V_)
        vTest = TestFunction(V_)
        a_proj = inner(dv, vTest)*dx
        b_proj = inner(v, vTest)*dx
        loc_solver = LocalSolver(a_proj, b_proj)
        loc_solver.factorize()
        if uOut is None:
            uOut = Function(V_)
            loc_solver.solve_local_rhs(uOut)
            return uOut
        else:
            loc_solver.solve_local_rhs(uOut)
            return

    corners = [
       (0.0, 0.0, 0.0),
       (0.0, W,   0.0),
       (0.0, 0.0, D),
       (0.0, W,   D)
    ]
    u_corners = np.zeros((Nsteps+1, len(corners)))

    lst_graph_tstep = []

    V_p = VectorFunctionSpace(mesh, 'CG', 1)
    V_stress_nodes = TensorFunctionSpace(mesh, "CG", 1)

    p_function = Function(V_p)  
    initial_coordinates = mesh.coordinates()

    for i in range(Nsteps):
        dt_i = time[i+1] - time[i]
        t_now = time[i+1]
        print("Time: ", t_now)

        p.t = t_now - float(alpha_f)*dt_i

        # Solve
        res_assembled = assemble(L_form)
        bc.apply(res_assembled)
        solver.solve(K, u.vector(), res_assembled)

        # Update
        update_fields(u, u_old, v_old, a_old)

        # Project stress
        local_project(sigma(u), Vsig, sig)
        stress_nodewise = project(sig, V_stress_nodes)

        # Force boundary
        bcs_force = DirichletBC(V_p, p, boundary_subdomains, 3)
        bcs_force.apply(p_function.vector())
        p_nodal_values = p_function.vector().get_local().reshape(-1, 3)
        node_force_t = torch.tensor(p_nodal_values, dtype=torch.float)

        # -------------- EVALUATE U AT EACH VERTEX --------------
        # This is the main difference
        disp_evaluated = evaluate_u_at_vertices(u, mesh)  
        deformed_coordinates = initial_coordinates + disp_evaluated

        pos_t = torch.tensor(deformed_coordinates, dtype=torch.float)
        vel_t = torch.tensor(v_old.vector().get_local().reshape(-1, 3), dtype=torch.float)
        acc_t = torch.tensor(a_old.vector().get_local().reshape(-1, 3), dtype=torch.float)
        sig_t = torch.tensor(stress_nodewise.vector().get_local().reshape(-1, 9), dtype=torch.float)

        graph = Data(
            x=pos_t,
            edge_index=bidir_edge_index,
            x_vel_t=vel_t,
            y_acc_t=acc_t,
            y_sig_t=sig_t,
            x_node_force=node_force_t,
            x_bc=node_scalar,
            y_node_lumped_masses=lumped_masses_nodes,
            x_element_connectivity=mesh_connectivity,
            time=torch.tensor([t_now], dtype=torch.float)
        )
        lst_graph_tstep.append(graph)

        p.t = t_now

        # Tip displacement
        if MPI.comm_world.size == 1:
            u_tip[i+1] = u(L, W, 0.)[1]
            for j, cpt in enumerate(corners):
                u_corners[i+1, j] = u(*cpt)[1]

        E_elas = assemble(0.5*k(u_old, u_old))
        E_kin = assemble(0.5*m(v_old, v_old))
        E_damp += dt_i*assemble(c(v_old, v_old))
        E_tot = E_elas + E_kin + E_damp
        energies[i+1, :] = np.array([E_elas, E_kin, E_damp, E_tot])

    simulation_name = (f'L{L}_W{W}_D{D}_NL{NL}_NW{NW}_ND{ND}_'
                       f'E{E}_nu{nu}_rho{density}_'
                       f'em{eta_m_val}_ek{eta_k_val}_Pi{initial_force}_'
                       f'T{total_time}_Tc{cutoff_time_factor*total_time}_Nsteps{num_steps}')

    save_path = './Results'
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    save_path_graph = os.path.join(save_path, mode, 'graphs')
    save_path_plot_check = os.path.join(save_path, mode, 'plot_check')
    os.makedirs(save_path_graph, exist_ok=True)
    os.makedirs(save_path_plot_check, exist_ok=True)

    plot_file_path = os.path.join(save_path_plot_check, f'plot_{simulation_name}.png')
    graph_file_path = os.path.join(save_path_graph, f'graphs{simulation_name}.pt')

    plot_check(time, u_tip, energies, plot_file_path)

    if MPI.comm_world.size == 1 and MPI.comm_world.rank == 0:
        fig2, ax2 = plt.subplots()
        for j in range(len(corners)):
            ax2.plot(time, u_corners[:, j], label=f'Corner {j} (y-disp)')
        ax2.set_xlabel("Time")
        ax2.set_ylabel("Y-Displacement of Clamped Corners")
        ax2.set_title("Corners Displacement Over Time")
        ax2.legend()
        corner_plot_path = os.path.join(save_path_plot_check, f'corner_plot_{simulation_name}.png')
        plt.savefig(corner_plot_path)
        print("Corner plot saved to:", corner_plot_path)
        plt.show()

    return lst_graph_tstep, u_tip, u_corners


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Input positions (example data; replace with your full data)
grph_0 = lst_graphs[0]
grph_25 = lst_graphs[20]
positions = grph_0.x.numpy()
positions_future = grph_25.x.numpy()

# Input forces (example data; replace with your full data)
forces = grph_0.x_node_force.numpy()

# Fixed node indicators (example data; replace with your full data)
fixed_nodes = grph_0.x_bc.numpy()

# Visualization
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot positions of nodes
x, y, z = positions[:, 0], positions[:, 1], positions[:, 2]
ax.scatter(x, y, z, c='b', label='Nodes')

# Plot forces as arrows
for i in range(len(positions)):
    ax.quiver(
        positions[i, 0], positions[i, 1], positions[i, 2],  # Start point
        positions_future[i,0]-positions[i, 0], positions_future[i,1]-positions[i, 1], positions_future[i,2]-positions[i, 2],  # Direction
        color='r', label='Force' if i == 0 else ""  # Add label only once
    )

# Highlight fixed nodes
fixed_node_indices = np.where(fixed_nodes == 1)[0]
ax.scatter(
    positions[fixed_node_indices, 0],
    positions[fixed_node_indices, 1],
    positions[fixed_node_indices, 2],
    c='g', label='Fixed Nodes', s=50
)

# Labels and legend
ax.set_xlabel('X Position')
ax.set_ylabel('Y Position')
ax.set_zlabel('Z Position')
ax.legend()
ax.set_title("3D Visualization of Cantilever Beam")

plt.show()
